# Model ablation study

Here we make a copy from the `MONTE` and add multiple functions to perform the ablation study.

In [1]:
import pickle
import numpy as np
import pandas as pd
from scipy.stats import t
from itertools import combinations
from typing import List, Tuple, Optional
from statsmodels.stats.multitest import multipletests
from scipy.stats import norm

class Monte:
    def __init__(
        self, alpha: float = 0.05, eps: float = 1e-10
    ):
        self.alpha = alpha
        self.eps = eps
        self.is_fitted: bool = False
        self.is_fine_tuned: bool = False
        self.coef_: pd.Series = pd.Series()  # (m,) tumor direction
        self.intercept_: pd.Series = pd.Series()  # (m,) baseline
        self.w_: pd.Series = pd.Series()  # (m,) probe weights (inverse noise)
        self.probe_ids: List = []
        self.best_top_n: Optional[int] = None
        self.best_tau2: Optional[float] = None

    # --------------------- FIT ---------------------
    def fit(
        self,
        X: pd.DataFrame,
        purity: pd.Series,
    ) -> "Monte":
        """
        X   : n x m methylation (rows=samples, cols=probes)
        purity: length-n purities in [0,1]
        """

        # Validate inputs
        self._validate_X(X)
        self._validate_purity(purity)

        
        X_arr = X.values.astype(float)
        p = np.asarray(purity, float).ravel()
        n, _ = X_arr.shape

        # Center
        X_mean = X_arr.mean(axis=0)  # (m,)
        p_mean = p.mean()
        Xc = X_arr - X_mean
        pc = p - p_mean

        # coef_
        denom = pc @ pc
        coef_ = (Xc.T @  pc) / max(denom, self.eps)  # (m,)

        if np.isnan(coef_).sum() > 0:
            raise RuntimeError("NaN values encountered in coefficient estimates.")

        #  intercept_
        intercept_ = X_mean - coef_ * p_mean  # (m,)

        # residuals and variances
        fitted = intercept_[None, :] + np.outer(p, coef_)  # (n, m)
        resid = X_arr - fitted  # (n, m)
        dof = max(n - 2, 1)
        sig2 = (resid**2).sum(axis=0) / dof

        # esitmate prior
        s0, d0 = self._estimate_prior_params(sig2)
        self.s0_, self.d0_ = s0, d0

        # posterior variances
        sig2_post = (d0 * s0 + dof * sig2) / (d0 + dof)

        # moderated t-statistics
        vbeta = 1.0 / (pc @ pc)
        t_raw = coef_ / np.sqrt(sig2 * vbeta)
        t_moderated = coef_ / np.sqrt(sig2_post * vbeta)
        df_total = d0 + dof

        # confidence intervals
        alpha = self.alpha
        t_crit = t.ppf(1 - alpha / 2, df_total)
        margin_of_error = t_crit * np.sqrt(sig2_post * vbeta)
        upper_CI = coef_ + margin_of_error
        lower_CI = coef_ - margin_of_error

        # test statistics p-values
        pvals = 2 * (1 - t.cdf(np.abs(t_moderated), df_total))
        _, p_adj, _, _ = multipletests(pvals, alpha=alpha, method="fdr_bh")

        # w = 1.0 / (sig2_used + 1e-6)                    # stable inverse-variance weights
        w = (coef_**2) / (sig2_post + 1e-6)  # SNR weights

        # Store
        self.intercept_ = pd.Series(intercept_, index=X.columns, name="intercept_")
        self.coef_ = pd.Series(coef_, index=X.columns, name="coef_")
        self.w_ = pd.Series(w, index=X.columns, name="weight")
        self.t_raw = pd.Series(t_raw, index=X.columns, name="t_raw")
        self.t_moderated = pd.Series(t_moderated, index=X.columns, name="t_moderated")
        self.df_total = df_total
        self.pvals = pd.Series(pvals, index=X.columns, name="p_value")
        self.p_adj = pd.Series(p_adj, index=X.columns, name="p_adj")
        self.probe_mean = pd.Series(X_mean, index=X.columns, name="probe_mean")
        self.residual_variance = pd.Series(
            sig2, index=X.columns, name="residual_variance"
        )
        self.moderated_variance = pd.Series(
            sig2_post, index=X.columns, name="moderated_variance"
        )
        self.purity_mean = p_mean
        self.probe_ids = list(X.columns)
        self.is_fitted = True
        self.df_stats = pd.DataFrame(
            {
                "intercept_": intercept_,
                "coef_": coef_,
                "upper_CI": upper_CI,
                "lower_CI": lower_CI,
                "weight": w,
                "t_raw": t_raw,
                "t_moderated": t_moderated,
                "p_value": pvals,
                "p_adj": p_adj,
            },
            index=X.columns,
        )

        return self

    def predict_purity(self, X: pd.DataFrame, top_n: Optional[int] = None) -> pd.Series:
        
        self._validate_X(X)
        self._check_is_fitted()

        selected_probes = self.probe_ids
        if top_n is not None:
            selected_probes = self.t_moderated.abs().nlargest(top_n).index.to_list()
        elif self.best_top_n is not None:
            selected_probes = (
                self.t_moderated.abs().nlargest(self.best_top_n).index.to_list()
            )

        X_arr = X.reindex(columns=selected_probes).values.astype(float)

        coef = np.asarray(self.coef_.reindex(index=selected_probes))
        w = np.asarray(self.w_.reindex(index=selected_probes))
        obs = np.isfinite(X_arr)

        # center by training mean
        Xc = (
            X_arr - self.probe_mean.reindex(index=selected_probes).values
        )  # store self.X_mean in fit()

        # weighted projection onto coef
        numerator = np.nansum(obs * w * Xc * coef, axis=1)
        denominator = np.nansum(obs * w * (coef * coef), axis=1) + self.eps
        p_hat = (
            numerator / denominator + self.purity_mean
        )  # add back training purity mean

        return pd.Series(
            np.clip(p_hat, 0.0, 1.0), index=X.index, name="predicted_purity"
        )
    
    def predict_purity_ols(self, X: pd.DataFrame, top_n: Optional[int] = None) -> pd.Series:
        
        self._validate_X(X)
        self._check_is_fitted()

        selected_probes = self.probe_ids
        if top_n is not None:
            selected_probes = self.t_moderated.abs().nlargest(top_n).index.to_list()
        elif self.best_top_n is not None:
            selected_probes = (
                self.t_moderated.abs().nlargest(self.best_top_n).index.to_list()
            )
        X_arr = X.reindex(columns=selected_probes).values.astype(float)

        coef = np.asarray(self.coef_.reindex(index=selected_probes))
        w = np.asarray(self.w_.reindex(index=selected_probes))
        obs = np.isfinite(X_arr)

        # center by training mean
        Xc = (
            X_arr - self.probe_mean.reindex(index=selected_probes).values
        )  # store self.X_mean in fit()

        # weighted projection onto coef
        numerator = np.nansum(obs * Xc * coef, axis=1)
        denominator = np.nansum(obs * (coef * coef), axis=1) + self.eps
        p_hat = (
            numerator / denominator + self.purity_mean
        )  # add back training purity mean

        return pd.Series(
            np.clip(p_hat, 0.0, 1.0), index=X.index, name="predicted_purity"
        )
    
    def predict_purity_var_weight(self, X: pd.DataFrame, top_n: Optional[int] = None) -> pd.Series:
        
        self._validate_X(X)
        self._check_is_fitted()

        selected_probes = self.probe_ids
        if top_n is not None:
            selected_probes = self.t_moderated.abs().nlargest(top_n).index.to_list()
        elif self.best_top_n is not None:
            selected_probes = (
                self.t_moderated.abs().nlargest(self.best_top_n).index.to_list()
            )
        X_arr = X.reindex(columns=selected_probes).values.astype(float)

        coef = np.asarray(self.coef_.reindex(index=selected_probes))
        w = 1 / np.asarray(self.moderated_variance.reindex(index=selected_probes))
        obs = np.isfinite(X_arr)

        # center by training mean
        Xc = (
            X_arr - self.probe_mean.reindex(index=selected_probes).values
        )  # store self.X_mean in fit()

        # weighted projection onto coef
        numerator = np.nansum(obs * w * Xc * coef, axis=1)
        denominator = np.nansum(obs * w * (coef * coef), axis=1) + self.eps
        p_hat = (
            numerator / denominator + self.purity_mean
        )  # add back training purity mean

        return pd.Series(
            np.clip(p_hat, 0.0, 1.0), index=X.index, name="predicted_purity"
        )

    @staticmethod
    def _estimate_prior_params(s2: np.ndarray) -> Tuple:
        from scipy.special import digamma

        """Empirical Bayes hyperparameters (limma-style)."""
        log_s2 = np.log(s2 + 1e-12)
        mean_log = np.mean(log_s2)
        var_log = np.var(log_s2, ddof=1)
        d0 = max(2.0 / var_log, 1.0)
        s0 = np.exp(mean_log - digamma(d0 / 2) + np.log(d0 / 2))
        return s0, d0

    def _validate_X(self, X: pd.DataFrame):
        """Validate input DataFrame X for NaN values."""
        if not isinstance(X, pd.DataFrame):
            raise ValueError("X must be a pandas DataFrame (samples x probes)")
        elif X.isnull().any().any():
            raise ValueError("X contains NaN values. Please handle missing data before using the model.")
    
    def _validate_purity(self, purity: pd.Series):
        """Validate input Series purity for NaN values."""
        if not isinstance(purity, pd.Series):
            raise ValueError("purity must be a pandas Series (samples,)")
        elif purity.isnull().any():
            raise ValueError("purity contains NaN values. Please handle missing data before using the model.")
        
    def _check_is_fitted(self):
        """Check if the model is fitted."""
        if not self.is_fitted:
            raise ValueError("Model has not been fitted yet. Fit the model before using this method.")

## Load data

**Training set**

In [2]:
from scipy.stats import pearsonr

In [3]:
train_metric="CPE"
compared_metric = "CPE"
top_n = 250 # this number is selected from the 01_pancancer_model_training.ipynb

In [ ]:
# the parquet file and metadata csv file could be downloaded from Zenodo
df_beta_train = pd.read_parquet("../../data/methylation/train_val_pan-cancer_beta.parquet")
df_meta_train = pd.read_csv("../../data/methylation/train_val_pan-cancer_meta.csv")
df_meta_train = df_meta_train.set_index("Barcode", drop=False)
df_meta_train = df_meta_train.loc[df_beta_train.index]

In [5]:
# filter to samples with purity values
df_meta_metric = df_meta_train.dropna(subset=[train_metric])
df_beta_metric = df_beta_train.loc[df_meta_metric["Barcode"]]

**Test set**

In [6]:
df_beta_test = pd.read_parquet("../../data/methylation/test_pan-cancer_beta.parquet")
df_meta_test = pd.read_csv("../../data/methylation/test_pan-cancer_meta.csv")
df_meta_test = df_meta_test.set_index("Barcode", drop=False)
df_meta_test = df_meta_test.loc[df_beta_test.index]

## Model training

In [7]:
model = Monte()
model.fit(df_beta_metric, df_meta_metric[train_metric])

## Comparison

In [8]:
results = []

for cancer in df_meta_test["Cancer.type"].unique():

    cancer_meta = df_meta_test[df_meta_test["Cancer.type"] == cancer]
    cancer_meta = cancer_meta.dropna(subset=[compared_metric])
    if len(cancer_meta) < 3:
        print(f"Skipping {cancer} due to insufficient samples with {compared_metric} values.")
        continue
    cancer_data = df_beta_test.loc[cancer_meta.index, :]
    
    ols_purity = model.predict_purity_ols(cancer_data)
    ols_topn_purity = model.predict_purity_ols(cancer_data, top_n=top_n)
    var_weight_purity = model.predict_purity_var_weight(cancer_data)
    var_weight_topn_purity = model.predict_purity_var_weight(cancer_data, top_n=top_n)
    monte_weight_purity = model.predict_purity(cancer_data)
    monte_weight_topn_purity = model.predict_purity(cancer_data, top_n=top_n)
    
    cor_ols, _ = pearsonr(cancer_meta[compared_metric], ols_purity)
    cor_ols_topn, _ = pearsonr(cancer_meta[compared_metric], ols_topn_purity)
    cor_var_weight, _ = pearsonr(cancer_meta[compared_metric], var_weight_purity)
    cor_monte_weight, _ = pearsonr(cancer_meta[compared_metric], monte_weight_purity)
    cor_var_weight_topn, _ = pearsonr(cancer_meta[compared_metric], var_weight_topn_purity)
    cor_monte_weight_topn, _ = pearsonr(cancer_meta[compared_metric], monte_weight_topn_purity)

    results.append([cancer, compared_metric, cor_ols, cor_ols_topn, cor_var_weight, cor_var_weight_topn, cor_monte_weight, cor_monte_weight_topn])
df_results = pd.DataFrame(results, columns=["Cancer.type", "Metric", "Cor_OLS", "Cor_OLS_TopN", "Cor_Var_Weight", "Cor_Var_Weight_TopN", "Cor_Monte_Weight", "Cor_Monte_Weight_TopN"])

**Statistical test**

In [9]:
from scipy.stats import wilcoxon

tests = [
    ("MONTE vs OLS", "Cor_Monte_Weight_TopN", "Cor_OLS"),
    ("MONTE vs Var", "Cor_Monte_Weight_TopN", "Cor_Var_Weight"),
    ("MONTE vs SNR", "Cor_Monte_Weight_TopN", "Cor_Monte_Weight"),
    ("SNR vs OLS", "Cor_Monte_Weight", "Cor_OLS"),
    ("SNR vs Var", "Cor_Monte_Weight", "Cor_Var_Weight"),
    ("Var vs OLS",   "Cor_Var_Weight",   "Cor_OLS"),
]

rows = []

df_results_nonan = df_results.dropna()
for name, col1, col2 in tests:
    stat, pval = wilcoxon(
        df_results_nonan[col1],
        df_results_nonan[col2],
        alternative="greater"
    )
    rows.append({
        "comparison": name,
        "method_1": col1,
        "method_2": col2,
        "statistic": stat,
        "p_value": pval
    })

df_wilcoxon = pd.DataFrame(rows)

In [10]:
df_wilcoxon

,comparison,method_1,method_2,statistic,p_value
0,MONTE vs OLS,Cor_Monte_Weight_TopN,Cor_OLS,230.0,9.536743e-07
1,MONTE vs Var,Cor_Monte_Weight_TopN,Cor_Var_Weight,229.0,1.430511e-06
2,MONTE vs SNR,Cor_Monte_Weight_TopN,Cor_Monte_Weight,189.0,4.508018e-03
3,SNR vs OLS,Cor_Monte_Weight,Cor_OLS,231.0,4.768372e-07
4,SNR vs Var,Cor_Monte_Weight,Cor_Var_Weight,231.0,4.768372e-07
5,Var vs OLS,Cor_Var_Weight,Cor_OLS,209.0,2.551079e-04


## Save results

In [11]:
df_results.to_csv(f"../../data/monte_outputs/pancancer/monte_ablation_correlation_results.csv", index=False)
df_wilcoxon.to_csv(f"../../data/monte_outputs/pancancer/monte_ablation_wilcoxon.csv", index=False)